# Eval Metric and Runtime Deltas

This notebook is intentionally split into stages:

1. Extract eval metrics from `eval_results/`
2. Inspect the parsed eval table locally
3. Query W&B Local for matching training runs and runtimes
4. Join both sources
5. Compute baseline-relative metric deltas and runtime percent deltas using `no_curriculum`


In [1]:
import json
import os
from pathlib import Path

import pandas as pd
import wandb


In [2]:
def resolve_repo_root():
    candidates = [Path.cwd(), Path.cwd().resolve(), Path(".").resolve(), Path("..").resolve()]

    try:
        candidates.extend(Path(__file__).resolve().parents)
    except NameError:
        pass

    notebook_hint = Path("notebooks/eval_metric_runtime_deltas.ipynb").resolve()
    candidates.extend(notebook_hint.parents)

    for candidate in candidates:
        if (candidate / "eval_results").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError(
        f"Could not locate repo root from cwd={Path.cwd()}"
    )


ROOT = resolve_repo_root()
EVAL_ROOT = ROOT / "eval_results"
OUTPUT_CSV = ROOT / "notebooks" / "eval_metric_runtime_deltas.csv"

WANDB_BASE_URL = os.environ.get("WANDB_BASE_URL", "http://localhost:8080")
WANDB_ENTITY = os.environ.get("WANDB_ENTITY", "sebi")
WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "curriculum_moe")

METRIC_KEYS = {
    "arc": "accuracy",
    "sciq": "accuracy",
    "gsm8k": "accuracy_extracted_answer",
}

ROOT, EVAL_ROOT, OUTPUT_CSV

(PosixPath('/home/sebi/disertatie'),
 PosixPath('/home/sebi/disertatie/eval_results'),
 PosixPath('/home/sebi/disertatie/notebooks/eval_metric_runtime_deltas.csv'))

## 1. Extract Eval Results Only

This section only reads local files under `eval_results/`. No W&B calls happen yet.


In [3]:
def metric_key_for(metrics):
    benchmark = metrics["benchmark"]
    metric_key = METRIC_KEYS.get(benchmark)
    if metric_key is None or metric_key not in metrics:
        raise KeyError(
            f"Unsupported benchmark metric for {benchmark}: {sorted(metrics.keys())}"
        )
    return metric_key


def parse_run_name(run_name, benchmark):
    if run_name.startswith("base__"):
        return None

    split_token = f"_{benchmark}_"
    if split_token not in run_name:
        return None

    model_family, suffix = run_name.split(split_token, 1)
    curriculum = suffix[:-5] if suffix.endswith("_lora") else suffix
    return model_family, curriculum


EVAL_COLUMNS = [
    "benchmark",
    "model_family",
    "curriculum",
    "run_name",
    "metric_name",
    "metric_value",
    "num_examples",
    "checkpoint_path",
    "model_id",
    "metrics_path",
]


def load_eval_table():
    rows = []

    for metrics_path in sorted(EVAL_ROOT.glob("*/*/metrics.json")):
        metrics = json.loads(metrics_path.read_text())
        run_name = metrics_path.parent.name
        benchmark = metrics["benchmark"]
        parsed = parse_run_name(run_name, benchmark)

        if parsed is None:
            continue

        model_family, curriculum = parsed
        metric_key = metric_key_for(metrics)

        rows.append(
            {
                "benchmark": benchmark,
                "model_family": model_family,
                "curriculum": curriculum,
                "run_name": run_name,
                "metric_name": metric_key,
                "metric_value": metrics[metric_key],
                "num_examples": metrics.get("num_examples"),
                "checkpoint_path": metrics.get("checkpoint_path"),
                "model_id": metrics.get("model_id"),
                "metrics_path": str(metrics_path),
            }
        )

    eval_df = pd.DataFrame(rows, columns=EVAL_COLUMNS)
    if eval_df.empty:
        return eval_df

    return eval_df.sort_values(
        ["benchmark", "model_family", "curriculum"]
    ).reset_index(drop=True)


In [4]:
eval_df = load_eval_table()
print(f"Eval rows: {len(eval_df)}")
eval_df

Eval rows: 48


,benchmark,model_family,curriculum,run_name,metric_name,metric_value,num_examples,checkpoint_path,model_id,metrics_path
0,arc,gpt_oss,data_curriculum,gpt_oss_arc_data_curriculum_lora,accuracy,0.874573,1172,ckpt/gpt_oss_arc_data_curriculum_lora/final_ad...,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
1,arc,gpt_oss,full_curriculum,gpt_oss_arc_full_curriculum_lora,accuracy,0.893345,1172,ckpt/gpt_oss_arc_full_curriculum_lora/final_ad...,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
2,arc,gpt_oss,model_curriculum,gpt_oss_arc_model_curriculum_lora,accuracy,0.879693,1172,ckpt/gpt_oss_arc_model_curriculum_lora/final_a...,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
3,arc,gpt_oss,no_curriculum,gpt_oss_arc_no_curriculum_lora,accuracy,0.886519,1172,ckpt/gpt_oss_arc_no_curriculum_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...
4,arc,lfm2,data_curriculum,lfm2_arc_data_curriculum_lora,accuracy,0.830205,1172,ckpt/lfm2_arc_data_curriculum_lora/final_adapter,LiquidAI/LFM2-8B-A1B,/home/sebi/disertatie/eval_results/arc/lfm2_ar...
5,arc,lfm2,full_curriculum,lfm2_arc_full_curriculum_lora,accuracy,0.802048,1172,ckpt/lfm2_arc_full_curriculum_lora/final_adapter,LiquidAI/LFM2-8B-A1B,/home/sebi/disertatie/eval_results/arc/lfm2_ar...
6,arc,lfm2,model_curriculum,lfm2_arc_model_curriculum_lora,accuracy,0.831911,1172,ckpt/lfm2_arc_model_curriculum_lora/final_adapter,LiquidAI/LFM2-8B-A1B,/home/sebi/disertatie/eval_results/arc/lfm2_ar...
7,arc,lfm2,no_curriculum,lfm2_arc_no_curriculum_lora,accuracy,0.834471,1172,ckpt/lfm2_arc_no_curriculum_lora/final_adapter,LiquidAI/LFM2-8B-A1B,/home/sebi/disertatie/eval_results/arc/lfm2_ar...
8,arc,olmoe,data_curriculum,olmoe_arc_data_curriculum_lora,accuracy,0.670648,1172,ckpt/olmoe_arc_data_curriculum_lora/final_adapter,allenai/OLMoE-1B-7B-0924-Instruct,/home/sebi/disertatie/eval_results/arc/olmoe_a...
9,arc,olmoe,full_curriculum,olmoe_arc_full_curriculum_lora,accuracy,0.668942,1172,ckpt/olmoe_arc_full_curriculum_lora/final_adapter,allenai/OLMoE-1B-7B-0924-Instruct,/home/sebi/disertatie/eval_results/arc/olmoe_a...


In [5]:
eval_df.groupby(["benchmark", "model_family"])["curriculum"].agg(list)

benchmark  model_family
arc        gpt_oss         [data_curriculum, full_curriculum, model_curri...
           lfm2            [data_curriculum, full_curriculum, model_curri...
           olmoe           [data_curriculum, full_curriculum, model_curri...
           qwen            [data_curriculum, full_curriculum, model_curri...
gsm8k      gpt_oss         [data_curriculum, full_curriculum, model_curri...
           lfm2            [data_curriculum, full_curriculum, model_curri...
           olmoe           [data_curriculum, full_curriculum, model_curri...
           qwen            [data_curriculum, full_curriculum, model_curri...
sciq       gpt_oss         [data_curriculum, full_curriculum, model_curri...
           lfm2            [data_curriculum, full_curriculum, model_curri...
           olmoe           [data_curriculum, full_curriculum, model_curri...
           qwen            [data_curriculum, full_curriculum, model_curri...
Name: curriculum, dtype: object

## 2. Query W&B Local

This section queries W&B only after the eval table is already extracted and visible.

Set these first if needed:

- `WANDB_BASE_URL`
- `WANDB_API_KEY`
- `WANDB_ENTITY`
- `WANDB_PROJECT`


In [6]:
if WANDB_ENTITY == "REPLACE_ME":
    raise ValueError(
        "Set WANDB_ENTITY before running the W&B cells. "
        "Example: export WANDB_ENTITY=your-team-or-username"
    )

api = wandb.Api(overrides={"base_url": WANDB_BASE_URL}, timeout=60)
WANDB_BASE_URL, WANDB_ENTITY, WANDB_PROJECT

wandb: Loading settings from /home/sebi/.config/wandb/settings
wandb: [wandb.Api()] Loaded credentials for http://localhost:8080 from /home/sebi/.netrc.


('http://localhost:8080', 'sebi', 'curriculum_moe')

In [7]:
def load_wandb_runtime_table(eval_run_names):
    rows = []

    runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}")

    for run in runs:
        config = dict(run.config or {})
        summary = dict(run.summary or {})

        output_dir = config.get("output_dir")
        output_run_name = Path(output_dir).name if output_dir else None
        candidate_run_name = output_run_name or run.name

        if candidate_run_name not in eval_run_names and run.name not in eval_run_names:
            continue

        total_runtime_seconds = summary.get("_runtime")
        if total_runtime_seconds is None:
            total_runtime_seconds = summary.get("_wandb", {}).get("runtime")
        if total_runtime_seconds is None:
            total_runtime_seconds = summary.get("train_runtime")

        rows.append(
            {
                "run_name": candidate_run_name,
                "wandb_run_name": run.name,
                "wandb_run_id": run.id,
                "wandb_run_path": "/".join(run.path),
                "wandb_run_url": getattr(run, "url", None),
                "created_at": getattr(run, "created_at", None),
                "state": getattr(run, "state", None),
                "wandb_project": WANDB_PROJECT,
                "dataset_id": config.get("dataset_id"),
                "output_dir": output_dir,
                "total_runtime_seconds": total_runtime_seconds,
                "train_runtime_seconds": summary.get("train_runtime"),
            }
        )

    wandb_df = pd.DataFrame(rows)
    if wandb_df.empty:
        return wandb_df, pd.DataFrame(columns=["run_name", "num_wandb_matches"])

    duplicate_counts = (
        wandb_df.groupby("run_name")
        .size()
        .rename("num_wandb_matches")
        .reset_index()
        .query("num_wandb_matches > 1")
        .sort_values(["num_wandb_matches", "run_name"], ascending=[False, True])
        .reset_index(drop=True)
    )

    wandb_df = wandb_df.sort_values(["created_at", "wandb_run_id"]).drop_duplicates(
        subset="run_name", keep="last"
    )

    return wandb_df.reset_index(drop=True), duplicate_counts


In [8]:
wandb_df, duplicate_wandb_runs = load_wandb_runtime_table(set(eval_df["run_name"]))
print(f"Matched W&B rows: {len(wandb_df)}")
wandb_df.sort_values(["run_name"]).reset_index(drop=True)

Matched W&B rows: 48


,run_name,wandb_run_name,wandb_run_id,wandb_run_path,wandb_run_url,created_at,state,wandb_project,dataset_id,output_dir,total_runtime_seconds,train_runtime_seconds
0,gpt_oss_arc_data_curriculum_lora,gpt_oss_arc_data_curriculum_lora,ja3c5r0e,sebi/curriculum_moe/ja3c5r0e,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:53:00Z,finished,curriculum_moe,None,None,306,258.7201
1,gpt_oss_arc_full_curriculum_lora,gpt_oss_arc_full_curriculum_lora,mrvxkuey,sebi/curriculum_moe/mrvxkuey,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:58:09Z,finished,curriculum_moe,None,None,281,234.4147
2,gpt_oss_arc_model_curriculum_lora,gpt_oss_arc_model_curriculum_lora,4v9r53ir,sebi/curriculum_moe/4v9r53ir,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:48:19Z,finished,curriculum_moe,None,None,278,230.7889
3,gpt_oss_arc_no_curriculum_lora,gpt_oss_arc_no_curriculum_lora,42vl1ht0,sebi/curriculum_moe/42vl1ht0,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:42:57Z,finished,curriculum_moe,None,None,319,261.4414
4,gpt_oss_gsm8k_data_curriculum_lora,gpt_oss_gsm8k_data_curriculum_lora,dx7gdqj8,sebi/curriculum_moe/dx7gdqj8,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:08:42Z,finished,curriculum_moe,None,None,1016,969.4363
5,gpt_oss_gsm8k_full_curriculum_lora,gpt_oss_gsm8k_full_curriculum_lora,srxbyczu,sebi/curriculum_moe/srxbyczu,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:25:42Z,finished,curriculum_moe,None,None,1031,983.3806
6,gpt_oss_gsm8k_model_curriculum_lora,gpt_oss_gsm8k_model_curriculum_lora,l14ane21,sebi/curriculum_moe/l14ane21,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T18:52:15Z,finished,curriculum_moe,None,None,984,936.7877
7,gpt_oss_gsm8k_no_curriculum_lora,gpt_oss_gsm8k_no_curriculum_lora,nz4273p1,sebi/curriculum_moe/nz4273p1,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T18:32:55Z,finished,curriculum_moe,None,None,1155,1108.8560
8,gpt_oss_sciq_data_curriculum_lora,gpt_oss_sciq_data_curriculum_lora,oddi7u8x,sebi/curriculum_moe/oddi7u8x,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T20:40:31Z,finished,curriculum_moe,None,None,1190,1143.4221
9,gpt_oss_sciq_full_curriculum_lora,gpt_oss_sciq_full_curriculum_lora,9qp97jay,sebi/curriculum_moe/9qp97jay,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T21:00:25Z,finished,curriculum_moe,None,None,1070,1019.7304


In [9]:
duplicate_wandb_runs

,run_name,num_wandb_matches


## 3. Join Eval Metrics with W&B Runtimes


In [10]:
comparison_table = eval_df.merge(
    wandb_df,
    on="run_name",
    how="left",
    validate="one_to_one",
)

print(f"Rows missing matched runtime: {comparison_table['total_runtime_seconds'].isna().sum()}")
comparison_table.sort_values(["benchmark", "model_family", "curriculum"]).reset_index(drop=True)

Rows missing matched runtime: 0


,benchmark,model_family,curriculum,run_name,metric_name,metric_value,num_examples,checkpoint_path,model_id,metrics_path,...,wandb_run_id,wandb_run_path,wandb_run_url,created_at,state,wandb_project,dataset_id,output_dir,total_runtime_seconds,train_runtime_seconds
0,arc,gpt_oss,data_curriculum,gpt_oss_arc_data_curriculum_lora,accuracy,0.874573,1172,ckpt/gpt_oss_arc_data_curriculum_lora/final_ad...,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,ja3c5r0e,sebi/curriculum_moe/ja3c5r0e,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:53:00Z,finished,curriculum_moe,None,None,306,258.7201
1,arc,gpt_oss,full_curriculum,gpt_oss_arc_full_curriculum_lora,accuracy,0.893345,1172,ckpt/gpt_oss_arc_full_curriculum_lora/final_ad...,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,mrvxkuey,sebi/curriculum_moe/mrvxkuey,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:58:09Z,finished,curriculum_moe,None,None,281,234.4147
2,arc,gpt_oss,model_curriculum,gpt_oss_arc_model_curriculum_lora,accuracy,0.879693,1172,ckpt/gpt_oss_arc_model_curriculum_lora/final_a...,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,4v9r53ir,sebi/curriculum_moe/4v9r53ir,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:48:19Z,finished,curriculum_moe,None,None,278,230.7889
3,arc,gpt_oss,no_curriculum,gpt_oss_arc_no_curriculum_lora,accuracy,0.886519,1172,ckpt/gpt_oss_arc_no_curriculum_lora/final_adapter,openai/gpt-oss-20b,/home/sebi/disertatie/eval_results/arc/gpt_oss...,...,42vl1ht0,sebi/curriculum_moe/42vl1ht0,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:42:57Z,finished,curriculum_moe,None,None,319,261.4414
4,arc,lfm2,data_curriculum,lfm2_arc_data_curriculum_lora,accuracy,0.830205,1172,ckpt/lfm2_arc_data_curriculum_lora/final_adapter,LiquidAI/LFM2-8B-A1B,/home/sebi/disertatie/eval_results/arc/lfm2_ar...,...,ebdfccv1,sebi/curriculum_moe/ebdfccv1,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T09:21:27Z,finished,curriculum_moe,None,None,191,180.7598
5,arc,lfm2,full_curriculum,lfm2_arc_full_curriculum_lora,accuracy,0.802048,1172,ckpt/lfm2_arc_full_curriculum_lora/final_adapter,LiquidAI/LFM2-8B-A1B,/home/sebi/disertatie/eval_results/arc/lfm2_ar...,...,k5kl6bgp,sebi/curriculum_moe/k5kl6bgp,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T09:24:41Z,finished,curriculum_moe,None,None,177,166.8936
6,arc,lfm2,model_curriculum,lfm2_arc_model_curriculum_lora,accuracy,0.831911,1172,ckpt/lfm2_arc_model_curriculum_lora/final_adapter,LiquidAI/LFM2-8B-A1B,/home/sebi/disertatie/eval_results/arc/lfm2_ar...,...,3d0e8ux3,sebi/curriculum_moe/3d0e8ux3,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T09:18:24Z,finished,curriculum_moe,None,None,179,169.0335
7,arc,lfm2,no_curriculum,lfm2_arc_no_curriculum_lora,accuracy,0.834471,1172,ckpt/lfm2_arc_no_curriculum_lora/final_adapter,LiquidAI/LFM2-8B-A1B,/home/sebi/disertatie/eval_results/arc/lfm2_ar...,...,vit5hp22,sebi/curriculum_moe/vit5hp22,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T09:15:08Z,finished,curriculum_moe,None,None,193,182.8723
8,arc,olmoe,data_curriculum,olmoe_arc_data_curriculum_lora,accuracy,0.670648,1172,ckpt/olmoe_arc_data_curriculum_lora/final_adapter,allenai/OLMoE-1B-7B-0924-Instruct,/home/sebi/disertatie/eval_results/arc/olmoe_a...,...,0r0ze9qh,sebi/curriculum_moe/0r0ze9qh,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-30T12:29:10Z,finished,curriculum_moe,None,None,145,135.0026
9,arc,olmoe,full_curriculum,olmoe_arc_full_curriculum_lora,accuracy,0.668942,1172,ckpt/olmoe_arc_full_curriculum_lora/final_adapter,allenai/OLMoE-1B-7B-0924-Instruct,/home/sebi/disertatie/eval_results/arc/olmoe_a...,...,mrzwrhiu,sebi/curriculum_moe/mrzwrhiu,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-30T12:31:39Z,finished,curriculum_moe,None,None,134,123.3258


## 4. Compute Baseline Deltas

Baseline is the `no_curriculum` run within each `(benchmark, model_family)` group.


In [11]:
def absolute_delta(value, baseline):
    if pd.isna(value) or pd.isna(baseline):
        return pd.NA
    return value - baseline


def pct_delta(value, baseline):
    if pd.isna(value) or pd.isna(baseline) or baseline == 0:
        return pd.NA
    return (value - baseline) / baseline * 100.0


baseline_table = (
    comparison_table.loc[
        comparison_table["curriculum"] == "no_curriculum",
        ["benchmark", "model_family", "metric_value", "total_runtime_seconds"],
    ]
    .rename(
        columns={
            "metric_value": "baseline_metric_value",
            "total_runtime_seconds": "baseline_total_runtime_seconds",
        }
    )
)

baseline_table

,benchmark,model_family,baseline_metric_value,baseline_total_runtime_seconds
3,arc,gpt_oss,0.886519,319
7,arc,lfm2,0.834471,193
11,arc,olmoe,0.666382,148
15,arc,qwen,0.781570,263
19,gsm8k,gpt_oss,0.712661,1155
23,gsm8k,lfm2,0.609553,624
27,gsm8k,olmoe,0.351782,458
31,gsm8k,qwen,0.417741,940
35,sciq,gpt_oss,0.960000,1186
39,sciq,lfm2,0.956000,801


In [12]:
comparison_table = comparison_table.merge(
    baseline_table,
    on=["benchmark", "model_family"],
    how="left",
    validate="many_to_one",
)

comparison_table["metric_delta"] = comparison_table.apply(
    lambda row: absolute_delta(row["metric_value"], row["baseline_metric_value"]),
    axis=1,
)
comparison_table["runtime_delta_pct"] = comparison_table.apply(
    lambda row: pct_delta(row["total_runtime_seconds"], row["baseline_total_runtime_seconds"]),
    axis=1,
)
comparison_table["is_baseline"] = comparison_table["curriculum"].eq("no_curriculum")

comparison_table = comparison_table[
    [
        "benchmark",
        "model_family",
        "curriculum",
        "run_name",
        "metric_name",
        "metric_value",
        "baseline_metric_value",
        "metric_delta",
        "total_runtime_seconds",
        "baseline_total_runtime_seconds",
        "runtime_delta_pct",
        "is_baseline",
        "wandb_run_name",
        "wandb_run_id",
        "wandb_run_path",
        "wandb_run_url",
        "created_at",
        "state",
        "wandb_project",
        "dataset_id",
        "output_dir",
        "checkpoint_path",
        "metrics_path",
    ]
].sort_values(["benchmark", "model_family", "curriculum"]).reset_index(drop=True)

comparison_table

,benchmark,model_family,curriculum,run_name,metric_name,metric_value,baseline_metric_value,metric_delta,total_runtime_seconds,baseline_total_runtime_seconds,...,wandb_run_id,wandb_run_path,wandb_run_url,created_at,state,wandb_project,dataset_id,output_dir,checkpoint_path,metrics_path
0,arc,gpt_oss,data_curriculum,gpt_oss_arc_data_curriculum_lora,accuracy,0.874573,0.886519,-0.011945,306,319,...,ja3c5r0e,sebi/curriculum_moe/ja3c5r0e,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:53:00Z,finished,curriculum_moe,None,None,ckpt/gpt_oss_arc_data_curriculum_lora/final_ad...,/home/sebi/disertatie/eval_results/arc/gpt_oss...
1,arc,gpt_oss,full_curriculum,gpt_oss_arc_full_curriculum_lora,accuracy,0.893345,0.886519,0.006826,281,319,...,mrvxkuey,sebi/curriculum_moe/mrvxkuey,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:58:09Z,finished,curriculum_moe,None,None,ckpt/gpt_oss_arc_full_curriculum_lora/final_ad...,/home/sebi/disertatie/eval_results/arc/gpt_oss...
2,arc,gpt_oss,model_curriculum,gpt_oss_arc_model_curriculum_lora,accuracy,0.879693,0.886519,-0.006826,278,319,...,4v9r53ir,sebi/curriculum_moe/4v9r53ir,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:48:19Z,finished,curriculum_moe,None,None,ckpt/gpt_oss_arc_model_curriculum_lora/final_a...,/home/sebi/disertatie/eval_results/arc/gpt_oss...
3,arc,gpt_oss,no_curriculum,gpt_oss_arc_no_curriculum_lora,accuracy,0.886519,0.886519,0.000000,319,319,...,42vl1ht0,sebi/curriculum_moe/42vl1ht0,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:42:57Z,finished,curriculum_moe,None,None,ckpt/gpt_oss_arc_no_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/gpt_oss...
4,arc,lfm2,data_curriculum,lfm2_arc_data_curriculum_lora,accuracy,0.830205,0.834471,-0.004266,191,193,...,ebdfccv1,sebi/curriculum_moe/ebdfccv1,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T09:21:27Z,finished,curriculum_moe,None,None,ckpt/lfm2_arc_data_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/lfm2_ar...
5,arc,lfm2,full_curriculum,lfm2_arc_full_curriculum_lora,accuracy,0.802048,0.834471,-0.032423,177,193,...,k5kl6bgp,sebi/curriculum_moe/k5kl6bgp,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T09:24:41Z,finished,curriculum_moe,None,None,ckpt/lfm2_arc_full_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/lfm2_ar...
6,arc,lfm2,model_curriculum,lfm2_arc_model_curriculum_lora,accuracy,0.831911,0.834471,-0.002560,179,193,...,3d0e8ux3,sebi/curriculum_moe/3d0e8ux3,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T09:18:24Z,finished,curriculum_moe,None,None,ckpt/lfm2_arc_model_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/lfm2_ar...
7,arc,lfm2,no_curriculum,lfm2_arc_no_curriculum_lora,accuracy,0.834471,0.834471,0.000000,193,193,...,vit5hp22,sebi/curriculum_moe/vit5hp22,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T09:15:08Z,finished,curriculum_moe,None,None,ckpt/lfm2_arc_no_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/lfm2_ar...
8,arc,olmoe,data_curriculum,olmoe_arc_data_curriculum_lora,accuracy,0.670648,0.666382,0.004266,145,148,...,0r0ze9qh,sebi/curriculum_moe/0r0ze9qh,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-30T12:29:10Z,finished,curriculum_moe,None,None,ckpt/olmoe_arc_data_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/olmoe_a...
9,arc,olmoe,full_curriculum,olmoe_arc_full_curriculum_lora,accuracy,0.668942,0.666382,0.002560,134,148,...,mrzwrhiu,sebi/curriculum_moe/mrzwrhiu,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-30T12:31:39Z,finished,curriculum_moe,None,None,ckpt/olmoe_arc_full_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/olmoe_a...


In [13]:
display_table = comparison_table.copy()
for column in [
    "metric_value",
    "baseline_metric_value",
    "metric_delta",
]:
    display_table[column] = pd.to_numeric(display_table[column], errors="coerce").round(3)

for column in [
    "total_runtime_seconds",
    "baseline_total_runtime_seconds",
    "runtime_delta_pct",
]:
    display_table[column] = pd.to_numeric(display_table[column], errors="coerce").round(2)

display_table

,benchmark,model_family,curriculum,run_name,metric_name,metric_value,baseline_metric_value,metric_delta,total_runtime_seconds,baseline_total_runtime_seconds,...,wandb_run_id,wandb_run_path,wandb_run_url,created_at,state,wandb_project,dataset_id,output_dir,checkpoint_path,metrics_path
0,arc,gpt_oss,data_curriculum,gpt_oss_arc_data_curriculum_lora,accuracy,0.875,0.887,-0.012,306,319,...,ja3c5r0e,sebi/curriculum_moe/ja3c5r0e,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:53:00Z,finished,curriculum_moe,None,None,ckpt/gpt_oss_arc_data_curriculum_lora/final_ad...,/home/sebi/disertatie/eval_results/arc/gpt_oss...
1,arc,gpt_oss,full_curriculum,gpt_oss_arc_full_curriculum_lora,accuracy,0.893,0.887,0.007,281,319,...,mrvxkuey,sebi/curriculum_moe/mrvxkuey,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:58:09Z,finished,curriculum_moe,None,None,ckpt/gpt_oss_arc_full_curriculum_lora/final_ad...,/home/sebi/disertatie/eval_results/arc/gpt_oss...
2,arc,gpt_oss,model_curriculum,gpt_oss_arc_model_curriculum_lora,accuracy,0.880,0.887,-0.007,278,319,...,4v9r53ir,sebi/curriculum_moe/4v9r53ir,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:48:19Z,finished,curriculum_moe,None,None,ckpt/gpt_oss_arc_model_curriculum_lora/final_a...,/home/sebi/disertatie/eval_results/arc/gpt_oss...
3,arc,gpt_oss,no_curriculum,gpt_oss_arc_no_curriculum_lora,accuracy,0.887,0.887,0.000,319,319,...,42vl1ht0,sebi/curriculum_moe/42vl1ht0,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T19:42:57Z,finished,curriculum_moe,None,None,ckpt/gpt_oss_arc_no_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/gpt_oss...
4,arc,lfm2,data_curriculum,lfm2_arc_data_curriculum_lora,accuracy,0.830,0.834,-0.004,191,193,...,ebdfccv1,sebi/curriculum_moe/ebdfccv1,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T09:21:27Z,finished,curriculum_moe,None,None,ckpt/lfm2_arc_data_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/lfm2_ar...
5,arc,lfm2,full_curriculum,lfm2_arc_full_curriculum_lora,accuracy,0.802,0.834,-0.032,177,193,...,k5kl6bgp,sebi/curriculum_moe/k5kl6bgp,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T09:24:41Z,finished,curriculum_moe,None,None,ckpt/lfm2_arc_full_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/lfm2_ar...
6,arc,lfm2,model_curriculum,lfm2_arc_model_curriculum_lora,accuracy,0.832,0.834,-0.003,179,193,...,3d0e8ux3,sebi/curriculum_moe/3d0e8ux3,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T09:18:24Z,finished,curriculum_moe,None,None,ckpt/lfm2_arc_model_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/lfm2_ar...
7,arc,lfm2,no_curriculum,lfm2_arc_no_curriculum_lora,accuracy,0.834,0.834,0.000,193,193,...,vit5hp22,sebi/curriculum_moe/vit5hp22,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-31T09:15:08Z,finished,curriculum_moe,None,None,ckpt/lfm2_arc_no_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/lfm2_ar...
8,arc,olmoe,data_curriculum,olmoe_arc_data_curriculum_lora,accuracy,0.671,0.666,0.004,145,148,...,0r0ze9qh,sebi/curriculum_moe/0r0ze9qh,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-30T12:29:10Z,finished,curriculum_moe,None,None,ckpt/olmoe_arc_data_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/olmoe_a...
9,arc,olmoe,full_curriculum,olmoe_arc_full_curriculum_lora,accuracy,0.669,0.666,0.003,134,148,...,mrzwrhiu,sebi/curriculum_moe/mrzwrhiu,http://localhost:8080/sebi/curriculum_moe/runs...,2026-03-30T12:31:39Z,finished,curriculum_moe,None,None,ckpt/olmoe_arc_full_curriculum_lora/final_adapter,/home/sebi/disertatie/eval_results/arc/olmoe_a...


In [14]:
reduced_table = display_table[[
    "benchmark",
    "model_family",
    "curriculum",
    "metric_value",
    "metric_delta",
    "total_runtime_seconds",
    "runtime_delta_pct",
]]

reduced_table["metric_value"] = reduced_table.apply(
    lambda row: (
        f"{row['metric_value']:.3f} ({row['metric_delta']:+.3f})"
        if (row["curriculum"] != "no_curriculum" and not pd.isna(row['metric_delta']))
        else f"{row['metric_value']:.3f}"
    ),
    axis=1,
)

reduced_table["total_runtime_seconds"] = reduced_table.apply(
    lambda row: (
        f"{row['total_runtime_seconds']}s ({row['runtime_delta_pct']:+.2f}%)"
        if (row["curriculum"] != "no_curriculum" and not pd.isna(row['runtime_delta_pct']))
        else f"{row['total_runtime_seconds']}s"
    ),
    axis=1,
)

reduced_table = reduced_table.drop(columns=["metric_delta", "runtime_delta_pct"])

reduced_table

reduced_table["curriculum_sort_key"] = reduced_table["curriculum"].apply(lambda x: 0 if x == "no_curriculum" else 1)
reduced_table = reduced_table.sort_values(["benchmark", "model_family", "curriculum_sort_key", "curriculum"]).drop(columns=["curriculum_sort_key"]).reset_index(drop=True)
reduced_table

,benchmark,model_family,curriculum,metric_value,total_runtime_seconds
0,arc,gpt_oss,no_curriculum,0.887,319s
1,arc,gpt_oss,data_curriculum,0.875 (-0.012),306s (-4.08%)
2,arc,gpt_oss,full_curriculum,0.893 (+0.007),281s (-11.91%)
3,arc,gpt_oss,model_curriculum,0.880 (-0.007),278s (-12.85%)
4,arc,lfm2,no_curriculum,0.834,193s
5,arc,lfm2,data_curriculum,0.830 (-0.004),191s (-1.04%)
6,arc,lfm2,full_curriculum,0.802 (-0.032),177s (-8.29%)
7,arc,lfm2,model_curriculum,0.832 (-0.003),179s (-7.25%)
8,arc,olmoe,no_curriculum,0.666,148s
9,arc,olmoe,data_curriculum,0.671 (+0.004),145s (-2.03%)


In [15]:
curriculum_order = [
    "no_curriculum",
    "model_curriculum",
    "data_curriculum",
    "full_curriculum",
]
available_curricula = [
    curriculum
    for curriculum in curriculum_order
    if curriculum in set(reduced_table["curriculum"].dropna())
]

paper_table = reduced_table.pivot(
    index=["benchmark", "model_family"],
    columns="curriculum",
    values=["metric_value", "total_runtime_seconds"],
)

paper_table = paper_table.reindex(
    columns=pd.MultiIndex.from_product(
        [["metric_value", "total_runtime_seconds"], available_curricula]
    )
)
paper_table = paper_table.swaplevel(0, 1, axis=1)
paper_table = paper_table.rename(
    columns={"metric_value": "Metric (Δ)", "total_runtime_seconds": "Runtime (Δ%)"},
    level=1,
)
paper_table.columns = pd.MultiIndex.from_tuples(
    [
        (
            curriculum.replace("_", " ").title(),
            stat,
        )
        for curriculum, stat in paper_table.columns
    ]
)

metric_columns = [(curriculum.replace("_", " ").title(), "Metric (Δ)") for curriculum in available_curricula]
runtime_columns = [(curriculum.replace("_", " ").title(), "Runtime (Δ%)") for curriculum in available_curricula]


def parse_primary_number(value):
    if pd.isna(value):
        return None
    token = str(value).strip().split(" ", 1)[0].rstrip("s")
    try:
        return float(token)
    except ValueError:
        return None


def highlight_best_per_row(row):
    styles = pd.Series("", index=row.index)

    metric_values = {
        column: parse_primary_number(row[column])
        for column in metric_columns
        if column in row.index
    }
    metric_values = {column: value for column, value in metric_values.items() if value is not None}
    if metric_values:
        best_metric = max(metric_values.values())
        for column, value in metric_values.items():
            if value == best_metric:
                styles[column] = "background-color: #56ad34; font-weight: 600"

    runtime_values = {
        column: parse_primary_number(row[column])
        for column in runtime_columns
        if column in row.index
    }
    runtime_values = {column: value for column, value in runtime_values.items() if value is not None}
    if runtime_values:
        best_runtime = min(runtime_values.values())
        for column, value in runtime_values.items():
            if value == best_runtime:
                styles[column] = "background-color: #578aba; font-weight: 600"

    return styles


(
    paper_table.style.set_caption("Grouped eval metric and runtime deltas")
    .apply(highlight_best_per_row, axis=1)
    .set_table_styles([
        {"selector": "caption", "props": [("caption-side", "bottom"), ("font-size", "0.95em")]},
        {"selector": "th.col_heading.level0", "props": [("text-align", "center"), ("border-bottom", "1px solid #000")]},
        {"selector": "th.col_heading.level1", "props": [("text-align", "center")]},
        {"selector": "th.row_heading", "props": [("text-align", "left")]},
    ])
    .set_properties(**{"text-align": "center", "white-space": "nowrap"})
)
